In [15]:
# First, install the psycopg2 package
!pip install psycopg2-binary

# Then import the required libraries
import http.client
import json
import pandas as pd
import csv
import psycopg2

In [4]:
import http.client

conn = http.client.HTTPSConnection("realty-in-us.p.rapidapi.com")

payload = "{\"limit\":200,\"offset\":0,\"postal_code\":\"90004\",\"status\":[\"for_sale\",\"ready_to_build\"],\"sort\":{\"direction\":\"desc\",\"field\":\"list_date\"}}"

headers = {
    'x-rapidapi-key': "cbdecb405bmsh2b2df9a44a9e1bbp1d3503jsnd30c22391c31",
    'x-rapidapi-host': "realty-in-us.p.rapidapi.com",
    'Content-Type': "application/json"
}

conn.request("POST", "/properties/v3/list", payload, headers)

res = conn.getresponse()
data = res.read()

# print(data.decode("utf-8"))
# Correct filename as a string and use write mode
filename = 'propertyRecord.json'

# Decode bytes to JSON first
data_json = json.loads(data.decode("utf-8"))

with open(filename, 'w', encoding='utf-8') as file:
    json.dump(data_json, file, indent=4)

In [5]:
# Load JSON
with open('propertyRecord.json', 'r', encoding='utf-8') as f:
    data_json = json.load(f)

# Extract the list of properties
results = data_json.get('data', {}).get('home_search', {}).get('results', [])

# Convert to DataFrame
propertyRecord_df = pd.json_normalize(results)  #flattens nested fields

# Show first rows
propertyRecord_df.head()

,__typename,property_id,listing_id,plan_id,status,photo_count,branding,open_houses,virtual_tours,matterport,...,lead_attributes.opcity_lead_attributes.flip_the_market_enabled,products.__typename,products.brand_name,products.products,estimate.__typename,estimate.estimate,pet_policy.__typename,pet_policy.cats,pet_policy.dogs,location.address.coordinate
0,SearchHome,1153571241,2992987794,None,for_sale,3,"[{'__typename': 'Branding', 'photo': None, 'na...",None,None,False,...,True,ProductSummarySearchHome,basic_opt_in,"[core.agent, co_broke]",NaN,NaN,NaN,NaN,NaN,NaN
1,SearchHome,2852214745,2992977247,None,for_sale,14,"[{'__typename': 'Branding', 'photo': None, 'na...",None,None,False,...,True,ProductSummarySearchHome,basic_opt_in,[co_broke],NaN,NaN,NaN,NaN,NaN,NaN
2,SearchHome,2213059567,2992970174,None,for_sale,31,"[{'__typename': 'Branding', 'photo': None, 'na...","[{'__typename': 'HomeOpenHouse', 'start_date':...",None,False,...,True,ProductSummarySearchHome,essentials,"[core.agent, core.broker, co_broke]",LatestEstimate,3907900.0,NaN,NaN,NaN,NaN
3,SearchHome,9890858939,2992894527,None,for_sale,45,"[{'__typename': 'Branding', 'photo': None, 'na...","[{'__typename': 'HomeOpenHouse', 'start_date':...",None,True,...,True,ProductSummarySearchHome,basic_opt_in,"[core.agent, co_broke]",LatestEstimate,1482600.0,NaN,NaN,NaN,NaN
4,SearchHome,2146837987,2992891255,None,for_sale,7,"[{'__typename': 'Branding', 'photo': None, 'na...",None,None,False,...,True,ProductSummarySearchHome,essentials,"[core.agent, core.broker, co_broke]",NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
propertyRecord_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 135 entries, 0 to 134
Data columns (total 90 columns):
 #   Column                                                          Non-Null Count  Dtype  
---  ------                                                          --------------  -----  
 0   __typename                                                      135 non-null    object 
 1   property_id                                                     135 non-null    object 
 2   listing_id                                                      135 non-null    object 
 3   plan_id                                                         0 non-null      object 
 4   status                                                          135 non-null    object 
 5   photo_count                                                     135 non-null    int64  
 6   branding                                                        135 non-null    object 
 7   open_houses                                          

In [7]:
print(propertyRecord_df.columns.tolist())

['__typename', 'property_id', 'listing_id', 'plan_id', 'status', 'photo_count', 'branding', 'open_houses', 'virtual_tours', 'matterport', 'advertisers', 'pet_policy', 'community', 'href', 'list_price', 'list_price_min', 'list_price_max', 'price_reduced_amount', 'estimate', 'last_sold_date', 'list_date', 'last_sold_price', 'location.__typename', 'location.address.__typename', 'location.address.city', 'location.address.line', 'location.address.street_name', 'location.address.street_number', 'location.address.street_suffix', 'location.address.country', 'location.address.postal_code', 'location.address.state_code', 'location.address.state', 'location.address.coordinate.__typename', 'location.address.coordinate.lat', 'location.address.coordinate.lon', 'location.address.coordinate.accuracy', 'location.street_view_url', 'location.county.__typename', 'location.county.fips_code', 'description.__typename', 'description.sub_type', 'description.type', 'description.beds', 'description.baths', 'desc

In [8]:
# Loop through all columns
for col in propertyRecord_df.columns:
    # If any row in the column is a dict or list
    if propertyRecord_df[col].apply(lambda x: isinstance(x, (dict, list))).any():
        propertyRecord_df[col] = propertyRecord_df[col].apply(lambda x: json.dumps(x) if x else None)

# Check result
propertyRecord_df.head()

,__typename,property_id,listing_id,plan_id,status,photo_count,branding,open_houses,virtual_tours,matterport,...,lead_attributes.opcity_lead_attributes.flip_the_market_enabled,products.__typename,products.brand_name,products.products,estimate.__typename,estimate.estimate,pet_policy.__typename,pet_policy.cats,pet_policy.dogs,location.address.coordinate
0,SearchHome,1153571241,2992987794,None,for_sale,3,"[{""__typename"": ""Branding"", ""photo"": null, ""na...",None,None,False,...,True,ProductSummarySearchHome,basic_opt_in,"[""core.agent"", ""co_broke""]",NaN,NaN,NaN,NaN,NaN,NaN
1,SearchHome,2852214745,2992977247,None,for_sale,14,"[{""__typename"": ""Branding"", ""photo"": null, ""na...",None,None,False,...,True,ProductSummarySearchHome,basic_opt_in,"[""co_broke""]",NaN,NaN,NaN,NaN,NaN,NaN
2,SearchHome,2213059567,2992970174,None,for_sale,31,"[{""__typename"": ""Branding"", ""photo"": null, ""na...","[{""__typename"": ""HomeOpenHouse"", ""start_date"":...",None,False,...,True,ProductSummarySearchHome,essentials,"[""core.agent"", ""core.broker"", ""co_broke""]",LatestEstimate,3907900.0,NaN,NaN,NaN,NaN
3,SearchHome,9890858939,2992894527,None,for_sale,45,"[{""__typename"": ""Branding"", ""photo"": null, ""na...","[{""__typename"": ""HomeOpenHouse"", ""start_date"":...",None,True,...,True,ProductSummarySearchHome,basic_opt_in,"[""core.agent"", ""co_broke""]",LatestEstimate,1482600.0,NaN,NaN,NaN,NaN
4,SearchHome,2146837987,2992891255,None,for_sale,7,"[{""__typename"": ""Branding"", ""photo"": null, ""na...",None,None,False,...,True,ProductSummarySearchHome,essentials,"[""core.agent"", ""core.broker"", ""co_broke""]",NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
# Fill ALL missing values at once
propertyRecord_df = propertyRecord_df.fillna({
    'list_price': 0,
    'description.beds': 0,
    'description.baths': 0,
    'description.sqft': 0,
    'location.address.city': 'Unknown',
    'location.address.state': 'Unknown'
})

# Drop columns that are completely empty
propertyRecord_df = propertyRecord_df.dropna(axis=1, how='all')

# Done
propertyRecord_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 135 entries, 0 to 134
Data columns (total 70 columns):
 #   Column                                                          Non-Null Count  Dtype  
---  ------                                                          --------------  -----  
 0   __typename                                                      135 non-null    object 
 1   property_id                                                     135 non-null    object 
 2   listing_id                                                      135 non-null    object 
 3   status                                                          135 non-null    object 
 4   photo_count                                                     135 non-null    int64  
 5   branding                                                        135 non-null    object 
 6   open_houses                                                     20 non-null     object 
 7   virtual_tours                                        

In [12]:
propertyRecord_df = propertyRecord_df.fillna({
    'price_reduced_amount': 0,
    'last_sold_price': 0,
    'description.baths_full': 0,
    'description.baths_half': 0,
    'description.baths_full_calc': 0,
    'description.baths_partial_calc': 0,
    'estimate.estimate': 0,
    'description.sub_type': 'Unknown',
    'flags.is_price_reduced': False,
    'flags.is_new_construction': False,
    'flags.is_contingent': False,
    'flags.is_pending': False
})

In [11]:
propertyRecord_df = propertyRecord_df.drop(columns=[
    '__typename',
    'location.__typename',
    'location.address.__typename',
    'location.address.coordinate.__typename',
    'location.county.__typename',
    'description.__typename',
    'flags.__typename',
    'source.__typename',
    'primary_photo.__typename',
    'estimate.__typename',
    'lead_attributes.__typename',
    'lead_attributes.opcity_lead_attributes.__typename',
    'products.__typename'
], errors='ignore')

In [13]:
propertyRecord_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 135 entries, 0 to 134
Data columns (total 57 columns):
 #   Column                                                          Non-Null Count  Dtype  
---  ------                                                          --------------  -----  
 0   property_id                                                     135 non-null    object 
 1   listing_id                                                      135 non-null    object 
 2   status                                                          135 non-null    object 
 3   photo_count                                                     135 non-null    int64  
 4   branding                                                        135 non-null    object 
 5   open_houses                                                     20 non-null     object 
 6   virtual_tours                                                   7 non-null      object 
 7   matterport                                           

In [14]:
# Drop columns with very few non-null values
columns_to_drop = [
    'virtual_tours', 
    'pet_policy.__typename', 
    'pet_policy.cats', 
    'pet_policy.dogs', 
    'open_houses'  # optional: if you consider it too sparse
]

# Drop them from the DataFrame
propertyRecord_df = propertyRecord_df.drop(columns=columns_to_drop)

# to Check the remaining columns
print(propertyRecord_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 135 entries, 0 to 134
Data columns (total 52 columns):
 #   Column                                                          Non-Null Count  Dtype  
---  ------                                                          --------------  -----  
 0   property_id                                                     135 non-null    object 
 1   listing_id                                                      135 non-null    object 
 2   status                                                          135 non-null    object 
 3   photo_count                                                     135 non-null    int64  
 4   branding                                                        135 non-null    object 
 5   matterport                                                      135 non-null    bool   
 6   advertisers                                                     135 non-null    object 
 7   href                                                 

In [16]:
# Select fact table columns
fact_columns = [
    'property_id', 'listing_id', 'list_price', 'price_reduced_amount',
    'last_sold_price', 'description.beds', 'description.baths', 'description.sqft',
    'description.lot_sqft', 'description.baths_full', 'description.baths_half',
    'description.baths_full_calc', 'description.baths_partial_calc', 'estimate.estimate'
]

# Create fact table
fact_table_df = propertyRecord_df[fact_columns].copy()


In [17]:
# ---------------------------
# 1. Location Dimension Table
# ---------------------------
location_dim = propertyRecord_df[[
    'location.address.city',
    'location.address.state',
    'location.address.country',
    'location.address.postal_code',
    'location.county.fips_code'
]].drop_duplicates().reset_index(drop=True)

# Add unique ID for joining with fact table
location_dim['location_id'] = range(1, len(location_dim)+1)

# -------------------------------
# 2. Property Type Dimension Table
# -------------------------------
property_type_dim = propertyRecord_df[[
    'description.type',
    'description.sub_type'
]].drop_duplicates().reset_index(drop=True)

# Add unique ID for joining with fact table
property_type_dim['property_type_id'] = range(1, len(property_type_dim)+1)

In [18]:
# Check Location Dimension
print("Location Dimension:")
print(location_dim.head())   # show first 5 rows
print(location_dim.info())   # show column info and types


Location Dimension:
  location.address.city location.address.state location.address.country  \
0           Los Angeles             California                      USA   
1          Silver Lakes             California                      USA   
2                Malibu             California                      USA   

  location.address.postal_code location.county.fips_code  location_id  
0                        90004                     06037            1  
1                        90004                     06037            2  
2                        90004                     06037            3  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 6 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   location.address.city         3 non-null      object
 1   location.address.state        3 non-null      object
 2   location.address.country      3 non-null      object
 

In [19]:
# Save Fact Table
fact_table_df.to_csv('fact_table.csv', index=False)  # index=False removes row numbers

# Save Location Dimension
location_dim.to_csv('location_dimension.csv', index=False)

# Save Property Type Dimension
property_type_dim.to_csv('property_type_dimension.csv', index=False)

print("All tables saved as CSV successfully!")

All tables saved as CSV successfully!


In [ ]:
Loading Layer

In [21]:
import psycopg2

# -------------------
# Database Connection
# -------------------
def get_db_connection():
    try:
        conn = psycopg2.connect(
            host="localhost",
            database="Realty_db",
            user="postgres",
            password="1234",
            port=5432
        )
        cursor = conn.cursor()
        print("Connection successful!")
        cursor.execute("SET search_path TO real_estate;")  # ensures schema is used
        return conn, cursor
    except Exception as e:
        print("Error connecting to database:", e)
        return None, None

# -------------------
# 2️ Rename DataFrame columns & convert types
# -------------------
# Location Dimension
location_dim = location_dim.rename(columns={
    'location.address.city': 'city',
    'location.address.state': 'state',
    'location.address.country': 'country',
    'location.address.postal_code': 'postal_code',
    'location.county.fips_code': 'fips_code'
}).astype({
    'city': str,
    'state': str,
    'country': str,
    'postal_code': str,
    'fips_code': str
})

# Property Type Dimension
property_type_dim = property_type_dim.rename(columns={
    'description.type': 'description_type',
    'description.sub_type': 'description_sub_type'
}).astype({
    'description_type': str,
    'description_sub_type': str
})

# Fact Table
fact_table_df = fact_table_df.rename(columns={
    'description.beds': 'beds',
    'description.baths': 'baths',
    'description.sqft': 'sqft'
}).astype({
    'property_id': str,
    'listing_id': str,
    'list_price': float,
    'beds': float,
    'baths': float,
    'sqft': float
})



In [22]:
# -------------------
# 3️ Create tables
# -------------------
def create_tables():
    conn, cursor = get_db_connection()
    if conn is None:
        return

    try:
        cursor.execute("""
        CREATE SCHEMA IF NOT EXISTS real_estate;

        DROP TABLE IF EXISTS real_estate.fact_property;
        DROP TABLE IF EXISTS real_estate.property_type_dim;
        DROP TABLE IF EXISTS real_estate.location_dim;

        CREATE TABLE real_estate.location_dim (
            location_id SERIAL PRIMARY KEY,
            city VARCHAR(100),
            state VARCHAR(100),
            country VARCHAR(50),
            postal_code VARCHAR(20),
            fips_code VARCHAR(20)
        );

        CREATE TABLE real_estate.property_type_dim (
            property_type_id SERIAL PRIMARY KEY,
            description_type VARCHAR(100),
            description_sub_type VARCHAR(100)
        );

        CREATE TABLE real_estate.fact_property (
            fact_id SERIAL PRIMARY KEY,
            property_id VARCHAR(50),
            listing_id VARCHAR(50),
            list_price NUMERIC,
            beds NUMERIC,
            baths NUMERIC,
            sqft NUMERIC,
            location_id INT REFERENCES location_dim(location_id),
            property_type_id INT REFERENCES property_type_dim(property_type_id)
        );
        """)
        conn.commit()
        print("Tables created successfully!")
    except Exception as e:
        print("Error creating tables:", e)
        conn.rollback()
    finally:
        cursor.close()
        conn.close()



In [23]:
# -------------------
# 4️⃣ Insert 1 row per table
# -------------------
def load_one_row(location_dim, property_type_dim, fact_table_df):
    conn, cursor = get_db_connection()
    if conn is None:
        return

    try:
        # Insert 1 row into location_dim
        row = location_dim.iloc[0]
        cursor.execute("""
            INSERT INTO location_dim (city, state, country, postal_code, fips_code)
            VALUES (%s, %s, %s, %s, %s) RETURNING location_id
        """, (row['city'], row['state'], row['country'], row['postal_code'], row['fips_code']))
        location_id = int(cursor.fetchone()[0])

        # Insert 1 row into property_type_dim
        row = property_type_dim.iloc[0]
        cursor.execute("""
            INSERT INTO property_type_dim (description_type, description_sub_type)
            VALUES (%s, %s) RETURNING property_type_id
        """, (row['description_type'], row['description_sub_type']))
        property_type_id = int(cursor.fetchone()[0])

        # Insert 1 row into fact_property
        row = fact_table_df.iloc[0]
        cursor.execute("""
            INSERT INTO fact_property (
                property_id, listing_id, list_price, beds, baths, sqft, location_id, property_type_id
            ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        """, (
            row['property_id'], row['listing_id'], float(row['list_price']),
            float(row['beds']), float(row['baths']), float(row['sqft']),
            location_id, property_type_id
        ))

        conn.commit()
        print("Inserted 1 row per table successfully!")
    except Exception as e:
        print("Error inserting data:", e)
        conn.rollback()
    finally:
        cursor.close()
        conn.close()

# -------------------
# 5️⃣ Run
# -------------------
create_tables()
load_one_row(location_dim, property_type_dim, fact_table_df)

Connection successful!
Tables created successfully!
Connection successful!
Inserted 1 row per table successfully!


In [24]:
def load_data_to_csv(csv_path, table_name):
    conn, cursor = get_db_connection()
    if conn is None:
        return

    try:
        with open(csv_path, 'r', encoding='utf-8') as file:
            reader = csv.reader(file)
            headers = next(reader)  # skip header row

            for row in reader:
                # Replace empty strings with None
                row = [None if cell == '' else cell for cell in row]

                # Build placeholders
                placeholders = ', '.join(['%s'] * len(row))
                columns = ', '.join(headers)
                sql = f"INSERT INTO {table_name} ({columns}) VALUES ({placeholders})"
                cursor.execute(sql, row)

        conn.commit()
        print(f"Data from {csv_path} inserted into {table_name} successfully!")

    except Exception as e:
        print("Error inserting data:", e)
        conn.rollback()

    finally:
        cursor.close()
        conn.close()

In [25]:
# Path to your fact table CSV
fact_csv_path = "anaconda_projects/030ec7da-d154-46a4-89df-1d5c17d7b4ce/fact_table.csv"

# Call the loader function
load_data_to_csv(fact_csv_path, "fact_property")

Connection successful!
Error inserting data: [Errno 2] No such file or directory: 'anaconda_projects/030ec7da-d154-46a4-89df-1d5c17d7b4ce/fact_table.csv'


In [26]:
import os
print(os.getcwd())

/Users/pc/anaconda_projects/030ec7da-d154-46a4-89df-1d5c17d7b4ce


In [27]:
fact_csv_path = "/Users/pc/anaconda_projects/030ec7da-d154-46a4-89df-1d5c17d7b4ce/fact_table.csv"

In [32]:
import pandas as pd

# Load original CSV
df = pd.read_csv("/Users/pc/anaconda_projects/030ec7da-d154-46a4-89df-1d5c17d7b4ce/fact_table.csv")

# First, check which columns actually exist in the DataFrame
print("Available columns:", df.columns.tolist())

# Filter the columns_to_keep list to only include columns that exist in the DataFrame
columns_to_keep = [
    'property_id', 'listing_id', 'list_price', 'beds', 'baths', 'sqft',
    'lot_sqft', 'baths_full', 'baths_half', 'baths_full_calc', 'baths_partial_calc',
    'estimate', 'location_id', 'property_type_id'
]

# Only keep columns that actually exist in the DataFrame
available_columns = [col for col in columns_to_keep if col in df.columns]
fact_clean_df = df[available_columns]

# Save cleaned CSV
clean_csv_path = "/Users/pc/anaconda_projects/030ec7da-d154-46a4-89df-1d5c17d7b4ce/fact_table_clean.csv"
fact_clean_df.to_csv(clean_csv_path, index=False)

print("Clean fact table CSV created!")









Available columns: ['property_id', 'listing_id', 'list_price', 'price_reduced_amount', 'last_sold_price', 'description.beds', 'description.baths', 'description.sqft', 'description.lot_sqft', 'description.baths_full', 'description.baths_half', 'description.baths_full_calc', 'description.baths_partial_calc', 'estimate.estimate']
Clean fact table CSV created!


In [33]:
load_data_to_csv(clean_csv_path, "fact_property")

Connection successful!
Data from /Users/pc/anaconda_projects/030ec7da-d154-46a4-89df-1d5c17d7b4ce/fact_table_clean.csv inserted into fact_property successfully!


In [35]:
import pandas as pd

df = pd.read_csv("/Users/pc/anaconda_projects/030ec7da-d154-46a4-89df-1d5c17d7b4ce/location_dimension.csv")

df = df.rename(columns={
    'location.address.city': 'city',
    'location.address.state': 'state',
    'location.address.country': 'country',
    'location.address.postal_code': 'postal_code',
    'location.county.fips_code': 'fips_code'
})

clean_path = "/Users/pc/anaconda_projects/030ec7da-d154-46a4-89df-1d5c17d7b4ce/location_dimension_clean.csv"
df.to_csv(clean_path, index=False)

print("Location cleaned!")

Location cleaned!


In [36]:
df = pd.read_csv("/Users/pc/anaconda_projects/030ec7da-d154-46a4-89df-1d5c17d7b4ce/property_type_dimension.csv")

df = df.rename(columns={
    'description.type': 'description_type',
    'description.sub_type': 'description_sub_type'
})

clean_path = "/Users/pc/anaconda_projects/030ec7da-d154-46a4-89df-1d5c17d7b4ce/property_type_dimension_clean.csv"
df.to_csv(clean_path, index=False)

print("Property type cleaned!")

Property type cleaned!


In [40]:
import os
import pandas as pd
import sqlite3
from pathlib import Path

# Define file paths
base_dir = Path("...")  # Replace with your actual base directory
fact_table_path = base_dir / "fact_table_clean.csv"
location_csv_path = base_dir / "location_dimension_clean.csv"
property_type_csv_path = base_dir / "property_type_dimension_clean.csv"

# Function to check if file exists and insert data
def insert_data_from_csv(conn, csv_path, table_name):
    try:
        # Check if file exists
        if not os.path.exists(csv_path):
            print(f"Warning: File not found: {csv_path}")
            return False
            
        # If file exists, read and insert data
        df = pd.read_csv(csv_path)
        df.to_sql(table_name, conn, if_exists='replace', index=False)
        print(f"Successfully inserted data into {table_name} table")
        return True
        
    except Exception as e:
        print(f"Error inserting data into {table_name}: {e}")
        return False

# Function to create dimension tables from fact table
def create_dimension_from_fact(fact_df, dimension_cols, key_col, table_name, output_path):
    if all(col in fact_df.columns for col in dimension_cols + [key_col]):
        # Create dimension dataframe with unique values
        dim_df = fact_df[dimension_cols + [key_col]].drop_duplicates()
        
        # Save to CSV
        dim_df.to_csv(output_path, index=False)
        print(f"Created {table_name} CSV at {output_path}")
        return dim_df
    else:
        missing_cols = [col for col in dimension_cols + [key_col] if col not in fact_df.columns]
        print(f"Cannot create {table_name}: Missing columns {missing_cols}")
        return None

# Connect to database
try:
    conn = sqlite3.connect('your_database.db')
    print("Connection successful!")
    
    # Check if fact table exists
    if os.path.exists(fact_table_path):
        fact_df = pd.read_csv(fact_table_path)
        print("Fact table loaded successfully")
        
        # Create location dimension if needed
        if not os.path.exists(location_csv_path):
            print(f"Attempting to create location dimension from fact table...")
            location_df = create_dimension_from_fact(
                fact_df, 
                ['city', 'state', 'zip', 'neighborhood'],  # Adjust these column names to match your fact table
                'location_id', 
                'location_dimension',
                location_csv_path
            )
            if location_df is not None:
                location_df.to_sql('location_dimension', conn, if_exists='replace', index=False)
                print("Successfully inserted data into location_dimension table")
        else:
            insert_data_from_csv(conn, location_csv_path, 'location_dimension')
        
        # Create property type dimension if needed
        if not os.path.exists(property_type_csv_path):
            print(f"Attempting to create property type dimension from fact table...")
            property_type_df = create_dimension_from_fact(
                fact_df,
                ['property_type', 'year_built'],  # Adjust these column names to match your fact table
                'property_type_id',
                'property_type_dimension',
                property_type_csv_path
            )
            if property_type_df is not None:
                property_type_df.to_sql('property_type_dimension', conn, if_exists='replace', index=False)
                print("Successfully inserted data into property_type_dimension table")
        else:
            insert_data_from_csv(conn, property_type_csv_path, 'property_type_dimension')
    else:
        print(f"Error: Fact table not found at {fact_table_path}")
    
    conn.close()
    
except Exception as e:
    print(f"Database connection error: {e}")

Connection successful!
Error: Fact table not found at .../fact_table_clean.csv


In [41]:
import os

print(os.listdir("/Users/pc/anaconda_projects/030ec7da-d154-46a4-89df-1d5c17d7b4ce"))



['Zipco_pipeline.ipynb', 'property_type_dimension.csv', 'property_type_dimension_clean.csv', 'location_dimension_clean.csv', 'fact_table.csv', 'fact_table_clean.csv', 'propertyRecord.json', '.ipynb_checkpoints', 'your_database.db', 'location_dimension.csv']


In [44]:
conn, cursor = get_db_connection()

cursor.execute("""
TRUNCATE TABLE real_estate.fact_property;
TRUNCATE TABLE real_estate.location_dim CASCADE;
TRUNCATE TABLE real_estate.property_type_dim CASCADE;
""")

conn.commit()
cursor.close()
conn.close()

print("Tables cleared!")

Connection successful!
Tables cleared!


In [46]:
load_data_to_csv(
    "/Users/pc/anaconda_projects/030ec7da-d154-46a4-89df-1d5c17d7b4ce/location_dimension_clean.csv",
    "location_dim"
)

load_data_to_csv(
    "/Users/pc/anaconda_projects/030ec7da-d154-46a4-89df-1d5c17d7b4ce/property_type_dimension_clean.csv",
    "property_type_dim"
)

load_data_to_csv(
    "/Users/pc/anaconda_projects/030ec7da-d154-46a4-89df-1d5c17d7b4ce/fact_table_clean.csv",
    "fact_property"
)

Connection successful!
Data from /Users/pc/anaconda_projects/030ec7da-d154-46a4-89df-1d5c17d7b4ce/location_dimension_clean.csv inserted into location_dim successfully!
Connection successful!
Data from /Users/pc/anaconda_projects/030ec7da-d154-46a4-89df-1d5c17d7b4ce/property_type_dimension_clean.csv inserted into property_type_dim successfully!
Connection successful!
Data from /Users/pc/anaconda_projects/030ec7da-d154-46a4-89df-1d5c17d7b4ce/fact_table_clean.csv inserted into fact_property successfully!
